# 03 - Datasets, DataLoaders, and batches

**Goals:** inspect a CSV, fit preprocessing on training rows, and build batches with Dataset and DataLoader.

Before applying the updates from [02](02_autograd.ipynb) to real batches, organize the observations. The [data description](../data/README.md) explains this synthetic CSV. Run cells in order in a fresh kernel; each notebook supplies its own setup.

## Load and inspect the data

Import the tools and set a seed so the shuffled order is reproducible when the notebook is run from the beginning.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Small examples run efficiently on a laptop.


Locate the CSV whether Jupyter starts in the repository root or the `notebooks` directory. The assertion gives a useful message if neither location works.

In [ ]:
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent

data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."

df = pd.read_csv(data_path)
df.head()


The CSV assigns each row to a split: training learns parameters, validation guides choices, and test is reserved for the final assessment. For these binary labels, the label mean is the proportion labeled 1.

In [ ]:
features = ["feature_1", "feature_2", "feature_3"]
train = df[df["split"] == "train"]
validation = df[df["split"] == "validation"]
test = df[df["split"] == "test"]

df.groupby("split")["label"].agg(["size", "mean"])


## Fit preprocessing on training data

The scaler learns feature means and scales from the training split only. Applying those same values to validation and test data avoids using information from those splits during fitting.

In [ ]:
scaler = StandardScaler().fit(train[features])
print("training means used by scaler:", scaler.mean_)


Convert each split into tensors using the fitted scaler. Features are `float32`; classification labels are integer class indices stored as `long`.

In [ ]:
def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)
x_test, y_test = tensors(test)


Because the scaler was fit on the training split, its transformed feature means should be close to zero. Small differences from zero reflect numerical precision.

In [ ]:
print("transformed training feature means:", x_train.mean(dim=0))
print("training shapes:", x_train.shape, y_train.shape)


## Represent individual observations

A `Dataset` provides a length and retrieves one observation by index. This example stores tensors in memory; a larger dataset could load an item on demand in `__getitem__`.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.inputs[index], self.labels[index]


In [ ]:
training_data = TabularDataset(x_train, y_train)
print("number of observations:", len(training_data))
print("one item:", training_data[0])


Each item is a feature vector and its scalar label. For tensors already in memory, `TensorDataset` provides the same item-pair interface; we will use it in later notebooks.

In [ ]:
tensor_data = TensorDataset(x_train, y_train)
print("built-in dataset item:", tensor_data[0])


## Combine observations into batches

A `DataLoader` collects individual items into batches. With `shuffle=True`, each new iteration over the training loader uses a shuffled order; validation and test loaders typically do not need shuffling. `num_workers=0` loads data in the main process.

In [ ]:
train_loader = DataLoader(training_data, batch_size=64, shuffle=True, num_workers=0)


Create an iterator and take its first batch. The first dimension counts observations; the second dimension of `x_batch` counts features. Labels have one entry per observation.

In [ ]:
x_batch, y_batch = next(iter(train_loader))
print("batch shapes:", x_batch.shape, y_batch.shape)
print("batch dtypes:", x_batch.dtype, y_batch.dtype)


Iterating over the loader again starts a new shuffled pass. There are 420 training observations: six full batches of 64 and a final batch of 36. The default `drop_last=False` retains that smaller batch.

In [ ]:
print("batch sizes:", [len(y) for x, y in train_loader])


## Try it

1. Change batch size to 100 and predict the number and sizes of batches.

2. Change `shuffle` and inspect several batches. Why is shuffled training useful?

3. Explain why fitting the scaler on the entire CSV leaks information.

4. Modify the Dataset to return the row index along with each example.

**Next:** [Models and losses](04_models_and_losses.ipynb) maps a batch of feature rows to predictions and a scalar loss.